# Prueba de la entrega en un entorno limpio (T4)

Simula lo que hará el docente: un Colab/Kaggle **nuevo** con GPU T4, instalar **solo** `requirements.txt`, importar `solucion.py` tal cual (con `config.json` y `adapter/`) y llamar a `generar_sql`.

Se verifica cada límite del contrato:

| Requisito | Límite |
|---|---|
| Carga del modelo (`import solucion`) | ≤ 5 min |
| Tiempo por pregunta | ≤ 30 s |
| Llamadas a `ejecutar` por pregunta | ≤ 3 |
| Salida | una sola consulta `SELECT` / `WITH ... SELECT` |
| Generación | determinista |

Además se instala **MySQL 8.4.10 desde el binario oficial** (el enlace del enunciado) y se evalúan las 40 preguntas de la validación propia con `ejecutar`, como lo hará el evaluador.

**Kaggle:** *Accelerator: GPU T4 x2*, *Internet: On*, *Secrets*: `GITHUB_TOKEN`. **Colab:** entorno de ejecución T4; el token se pide al inicio.

In [ ]:
# 1. Repo + requirements.txt (y nada más)
import os, sys, subprocess, time
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle da 2 T4; el docente usa 1
EN_KAGGLE = os.path.exists("/kaggle/working")
REPO = "Albu231311/Deep-Proyecto3"
ROOT_DIR = Path("/kaggle/working/repo") if EN_KAGGLE else Path("/content/repo")

if not ROOT_DIR.exists():
    if EN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        import getpass
        token = getpass.getpass("GitHub token: ")
    r = subprocess.run(f"git clone -q https://{token}@github.com/{REPO}.git {ROOT_DIR}",
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo: " + r.stderr.replace(token, "***"))
os.chdir(ROOT_DIR)
sys.path.insert(0, str(ROOT_DIR))

t = time.time()
subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
print(f"requirements.txt instalado en {time.time() - t:.0f} s")
print(subprocess.run("git log -1 --format='commit %h — %s'", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. Carga del modelo tal como la hará el docente
import json, torch
print("config.json:", json.load(open("config.json")))
print("adapter/:", sorted(p.name for p in Path("adapter").iterdir()))
print("GPU:", torch.cuda.get_device_name(0))

t = time.time()
import solucion
CARGA_S = time.time() - t
print(f"\nimport solucion: {CARGA_S:.1f} s (límite 300 s) -> {'OK' if CARGA_S <= 300 else 'EXCEDE'}")

In [ ]:
# 3. Una pregunta sin base de datos (ejecutar=None)
esquema_demo = '''CREATE TABLE `clientes` (
  `id` int NOT NULL,
  `nombre` varchar(100) NOT NULL COMMENT 'Nombre completo',
  `ciudad` varchar(50) DEFAULT NULL COMMENT 'Ciudad de residencia',
  PRIMARY KEY (`id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='Clientes de la tienda'

CREATE TABLE `pedidos` (
  `id` int NOT NULL,
  `cliente_id` int NOT NULL COMMENT 'Cliente que hizo el pedido',
  `total` decimal(10,2) NOT NULL COMMENT 'Monto del pedido en USD',
  `fecha` date NOT NULL,
  PRIMARY KEY (`id`),
  CONSTRAINT `pedidos_ibfk_1` FOREIGN KEY (`cliente_id`) REFERENCES `clientes` (`id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='Pedidos'
'''
pregunta = "¿Cuánto ha gastado en total cada cliente de Guatemala? Devuelve las columnas: nombre, gasto_total."
t = time.time()
sql = solucion.generar_sql(esquema_demo, pregunta)
print(f"{time.time() - t:.1f} s\n{sql}")

## MySQL 8.4.10 (binario oficial del enunciado)

In [ ]:
# 4. Instalar y arrancar MySQL 8.4.10; usuarios root/root2026 y evaluador/eval2026 (src/mysql_utils.py)
SCRIPT = r'''#!/bin/bash
# Instala y arranca MySQL 8.4.10 (binario oficial) en Colab/Kaggle (Ubuntu, usuario root).
# Los usuarios se crean desde Python (el cliente `mysql` del binario minimal necesita
# libncurses5, que no existe en Ubuntu reciente).
set -e
BASE=/opt/mysql
DATOS=/opt/mysql-data
URL=https://downloads.mysql.com/archives/get/p/23/file/mysql-8.4.10-linux-glibc2.17-x86_64-minimal.tar.xz

if [ ! -x "$BASE/bin/mysqld" ]; then
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq libaio1 libnuma1 wget xz-utils >/dev/null 2>&1 || {
    apt-get install -y -qq libaio1t64 libnuma1 wget xz-utils >/dev/null 2>&1
    ln -sf /usr/lib/x86_64-linux-gnu/libaio.so.1t64 /usr/lib/x86_64-linux-gnu/libaio.so.1
  }
  wget -q -O /tmp/mysql.tar.xz "$URL"
  mkdir -p "$BASE"
  tar -xf /tmp/mysql.tar.xz -C "$BASE" --strip-components=1
fi

if [ ! -d "$DATOS/mysql" ]; then
  "$BASE/bin/mysqld" --initialize-insecure --user=root --basedir="$BASE" --datadir="$DATOS" >/dev/null 2>&1
fi

if ! "$BASE/bin/mysqladmin" -h127.0.0.1 -uroot -proot2026 ping >/dev/null 2>&1 && \
   ! "$BASE/bin/mysqladmin" -h127.0.0.1 -uroot ping >/dev/null 2>&1; then
  "$BASE/bin/mysqld" --user=root --basedir="$BASE" --datadir="$DATOS" \
    --port=3306 --bind-address=127.0.0.1 --socket=/tmp/mysql.sock \
    --log-error=/tmp/mysqld.log --daemonize
fi

for i in $(seq 1 30); do
  if "$BASE/bin/mysqladmin" -h127.0.0.1 -uroot -proot2026 ping >/dev/null 2>&1 || \
     "$BASE/bin/mysqladmin" -h127.0.0.1 -uroot ping >/dev/null 2>&1; then
    echo "mysqld 8.4.10 listo en 127.0.0.1:3306"
    exit 0
  fi
  sleep 1
done
echo "mysqld no respondió; ver /tmp/mysqld.log" >&2
exit 1'''
Path("/tmp/instalar_mysql.sh").write_text(SCRIPT)
t = time.time()
r = subprocess.run("bash /tmp/instalar_mysql.sh", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "MySQL no arrancó"

import mysql.connector

# root sin contraseña tras --initialize-insecure; si ya se configuró, root2026
try:
    conn = mysql.connector.connect(host="127.0.0.1", user="root", password="")
except mysql.connector.Error:
    conn = mysql.connector.connect(host="127.0.0.1", user="root", password="root2026")
cur = conn.cursor()
for stmt in [
    "ALTER USER 'root'@'localhost' IDENTIFIED BY 'root2026'",
    "CREATE USER IF NOT EXISTS 'root'@'%' IDENTIFIED BY 'root2026'",
    "GRANT ALL ON *.* TO 'root'@'%' WITH GRANT OPTION",
    "CREATE USER IF NOT EXISTS 'evaluador'@'%' IDENTIFIED BY 'eval2026'",
    "FLUSH PRIVILEGES",
]:
    cur.execute(stmt)
cur.execute("SELECT VERSION(), @@GLOBAL.sql_mode")
version, modo = cur.fetchone()
conn.close()
print(f"MySQL {version} | ONLY_FULL_GROUP_BY: {'ONLY_FULL_GROUP_BY' in modo}")
print(f"MySQL listo en {time.time() - t:.0f} s")

In [ ]:
# 5. Cargar los esquemas de validación
from src.mysql_utils import cargar_esquema

for nombre in ["sistema_academico", "clinica_salud"]:
    cargar_esquema(f"datos/validacion/esquemas_val/{nombre}.sql", nombre)
with open("datos/validacion/val.jsonl", encoding="utf-8") as f:
    val_preguntas = [json.loads(l) for l in f]
db_map = {"sistema_academico": "sistema_academico", "clinica_salud": "clinica_salud"}
print(len(val_preguntas), "preguntas")

## Evaluación completa con `ejecutar` (como el docente)

In [ ]:
# 6. 40 preguntas: tiempo, llamadas a ejecutar, solo lectura y EX
from src.evaluador import evaluar_validacion
from src.filtro import es_solo_lectura

registro = []

def generar_como_docente(esquema, pregunta, ej):
    llamadas = [0]
    def ejecutar(sql):
        llamadas[0] += 1
        return ej(sql)
    t = time.time()
    sql = solucion.generar_sql(esquema, pregunta, ejecutar)
    registro.append({"segundos": time.time() - t, "llamadas": llamadas[0],
                     "solo_lectura": es_solo_lectura(sql), "sql": sql})
    return sql

res = evaluar_validacion(val_preguntas, generar_como_docente, db_map)
print(res["resumen"])

In [ ]:
# 7. Determinismo: repetir 8 preguntas y comparar
from src.mysql_utils import obtener_esquema, hacer_ejecutar

distintas = 0
for p, reg in list(zip(val_preguntas, registro))[::5]:
    db = db_map[p["esquema_id"]]
    sql = solucion.generar_sql(obtener_esquema(db), p["pregunta"], hacer_ejecutar(db))
    distintas += sql != reg["sql"]
print(f"Determinismo: {8 - distintas}/8 idénticas")

In [ ]:
# 8. Resumen del contrato
import pandas as pd

tiempos = [r["segundos"] for r in registro]
checks = pd.DataFrame([
    ["Carga del modelo ≤ 300 s", f"{CARGA_S:.1f} s", CARGA_S <= 300],
    ["Tiempo máximo por pregunta ≤ 30 s", f"{max(tiempos):.1f} s (medio {sum(tiempos)/len(tiempos):.1f} s)", max(tiempos) <= 30],
    ["Llamadas a ejecutar ≤ 3", f"máx {max(r['llamadas'] for r in registro)}", max(r["llamadas"] for r in registro) <= 3],
    ["Siempre SELECT / WITH (solo lectura)", f"{sum(r['solo_lectura'] for r in registro)}/{len(registro)}", all(r["solo_lectura"] for r in registro)],
    ["Determinista", f"{8 - distintas}/8 idénticas", distintas == 0],
    ["EX ponderado (validación propia)", f"{res['ex_ponderado']:.4f}", None],
], columns=["Requisito", "Valor", "Cumple"])
checks